# SCUC - In-Class Example 4

**Original AMPL author:** Xingpeng Li - Associate Professor, Dept. of Electrical and Computer Engineering, University of Houston (UH), Houston, TX, USA (Senior Member, IEEE). Email: xli83@central.uh.edu

**Converted by:** Haoxiang Wan - PhD student of Dr. Xingpeng Li (AMPL -> Pyomo + Gurobi)

Two-unit, two-period UC with start-up indicators and ramp limits (Big-M).

In [1]:
from pyomo.environ import (
    ConcreteModel, Var, Objective, Constraint, SolverFactory,
    Binary, minimize, value
)

BigM = 1e3
c1, SU1, NL1, RR1, Pgmin1, Pgmax1 = 10, 800, 100, 25, 40, 80
c2, SU2, NL2, RR2, Pgmin2, Pgmax2 = 30, 100,  50, 30, 20, 90
TLoad_H1, TLoad_H2 = 70, 110

model = ConcreteModel()
for v in ['u1_H1','u1_H2','u2_H1','u2_H2','v1_H1','v1_H2','v2_H1','v2_H2']:
    setattr(model, v, Var(domain=Binary))
for v in ['Pg1_H1','Pg1_H2','Pg2_H1','Pg2_H2']:
    setattr(model, v, Var())

m = model
m.obj = Objective(
    expr=(c1*m.Pg1_H1 + c1*m.Pg1_H2 + c2*m.Pg2_H1 + c2*m.Pg2_H2)
       + (NL1*m.u1_H1 + NL1*m.u1_H2 + NL2*m.u2_H1 + NL2*m.u2_H2)
       + (SU1*m.v1_H1 + SU1*m.v1_H2 + SU2*m.v2_H1 + SU2*m.v2_H2),
    sense=minimize
)

m.PowerBalance_H1 = Constraint(expr=m.Pg1_H1 + m.Pg2_H1 == TLoad_H1)
m.PowerBalance_H2 = Constraint(expr=m.Pg1_H2 + m.Pg2_H2 == TLoad_H2)

m.genLimit_1_Min_H1 = Constraint(expr=Pgmin1*m.u1_H1 <= m.Pg1_H1)
m.genLimit_1_Min_H2 = Constraint(expr=Pgmin1*m.u1_H2 <= m.Pg1_H2)
m.genLimit_1_Max_H1 = Constraint(expr=m.Pg1_H1 <= Pgmax1*m.u1_H1)
m.genLimit_1_Max_H2 = Constraint(expr=m.Pg1_H2 <= Pgmax1*m.u1_H2)

m.genRRLimit_1_H2_Up = Constraint(expr=m.Pg1_H2 - m.Pg1_H1 <= RR1*m.u1_H1 + BigM*m.v1_H2)
m.genRRLimit_1_H2_Dn = Constraint(expr=m.Pg1_H1 - m.Pg1_H2 <= RR1*m.u1_H2 + BigM*(m.v1_H2 - m.u1_H2 + m.u1_H1))

m.genVU_1_H1 = Constraint(expr=m.v1_H1 >= m.u1_H1)
m.genVU_1_H2 = Constraint(expr=m.v1_H2 >= m.u1_H2 - m.u1_H1)

m.genLimit_2_Min_H1 = Constraint(expr=Pgmin2*m.u2_H1 <= m.Pg2_H1)
m.genLimit_2_Min_H2 = Constraint(expr=Pgmin2*m.u2_H2 <= m.Pg2_H2)
m.genLimit_2_Max_H1 = Constraint(expr=m.Pg2_H1 <= Pgmax2*m.u2_H1)
m.genLimit_2_Max_H2 = Constraint(expr=m.Pg2_H2 <= Pgmax2*m.u2_H2)

m.genRRLimit_2_H2_Up = Constraint(expr=m.Pg2_H2 - m.Pg2_H1 <= RR2*m.u2_H1 + BigM*m.v2_H2)
m.genRRLimit_2_H2_Dn = Constraint(expr=m.Pg2_H1 - m.Pg2_H2 <= RR2*m.u2_H2 + BigM*(m.v2_H2 - m.u2_H2 + m.u2_H1))

m.genVU_2_H1 = Constraint(expr=m.v2_H1 >= m.u2_H1)
m.genVU_2_H2 = Constraint(expr=m.v2_H2 >= m.u2_H2 - m.u2_H1)

In [2]:
# ---- Solve with Gurobi ----
solver = SolverFactory('gurobi')
solver.options['MIPGap'] = 0.0
solver.options['TimeLimit'] = 90
results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)
m = model
print(f"H1: v1={value(m.v1_H1)}, u1={value(m.u1_H1)}, Pg1={value(m.Pg1_H1):.3f}",
      f"v2={value(m.v2_H1)}, u2={value(m.u2_H1)}, Pg2={value(m.Pg2_H1):.3f}")
print(f"H2: v1={value(m.v1_H2)}, u1={value(m.u1_H2)}, Pg1={value(m.Pg1_H2):.3f}",
      f"v2={value(m.v2_H2)}, u2={value(m.u2_H2)}, Pg2={value(m.Pg2_H2):.3f}")

Read LP format model from file C:\Users\hwan6\AppData\Local\Temp\tmp9m3yubbk.pyomo.lp


Reading time = 0.00 seconds
x1: 18 rows, 12 columns, 48 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]


Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
TimeLimit  90


MIPGap  0

Optimize a model with 18 rows, 12 columns and 48 nonzeros


Model fingerprint: 0x78af52d6


...

Root relaxation: objective 3.550000e+03, 0 iterations, 0.00 seconds (0.00 work units)



    Nodes    |    Current Node    |     Objective Bounds      |     Work


 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    3550.0000000 3550.00000  0.00%     -    0s



Explored 1 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 20 (of 20 available processors)

Solution count 5: 3550 4100 4200 ... 4900

Optimal solution found (tolerance 0.00e+00)
Best objective 3.550000000000e+03, best bound 3.550000000000e+03, gap 0.0000%


ok optimal
H1: v1=1.0, u1=1.0, Pg1=70.000 v2=-0.0, u2=-0.0, Pg2=0.000
H2: v1=-0.0, u1=1.0, Pg1=80.000 v2=1.0, u2=1.0, Pg2=30.000
